In [ ]:
pip install wfdb scipy scikit-learn pandas numpy
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb

from scipy.signal import resample_poly
from sklearn.model_selection import train_test_split, GroupShuffleSplit


# =============================================================================
# Configuration
# =============================================================================

DATASET_DIR = Path(r"C:\repos\datasets\MIT-BIH")
OUTPUT_DIR = DATASET_DIR / "processed_250Hz_187"

DOWNLOAD_DATABASE = True

TARGET_FS = 250
N_SAMPLES = 187

# "MLII":
#   Use only true MLII channels.
#   Records 102 and 104 will be skipped because they contain no MLII.
#
# "first":
#   Always use channel 0.
#   This more closely matches ECG-LDC's reported preprocessing.
CHANNEL_MODE = "MLII"

# Options:
#   "none"
#   "per_beat_zscore"
#   "global_zscore"
#
# ECG-LDC does not report a separate z-score normalization step;
# its explicit input scaling is train-derived 8-bit min/max quantization.
NORMALIZATION = "none"

# Options:
#   "random"
#   "inter_patient"
#   "dechazal"
#
# random:
#   Stratified beat-wise split. Same patient can occur in all partitions.
#
# inter_patient:
#   Strict patient-independent random split.
#   Records 201 and 202 are correctly treated as the SAME subject.
#
# dechazal:
#   Canonical DS1/DS2 protocol used extensively in MIT-BIH literature.
#   Note: canonical DS1/DS2 places records 201 and 202 in different sets,
#   even though they are from the same patient.
SPLIT_MODE = "random"

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

RANDOM_SEED = 42

# Save X_all/y_all as well as the six requested split arrays.
# Useful because you can later rebuild different split protocols without
# rerunning WFDB/resampling.
SAVE_FULL_DATASET = True


# =============================================================================
# AAMI mapping
# =============================================================================

AAMI_MAP = {
    # N: Normal / bundle branch / escape
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",

    # S: Supraventricular ectopic
    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",

    # V: Ventricular ectopic
    "V": "V",
    "E": "V",

    # F: Fusion of normal and ventricular
    "F": "F",

    # Q: Unknown / paced
    "/": "Q",
    "f": "Q",
    "Q": "Q",
}

CLASS_TO_INDEX = {
    "N": 0,
    "S": 1,
    "V": 2,
    "F": 3,
    "Q": 4,
}

INDEX_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_INDEX.items()
}


# =============================================================================
# Canonical de Chazal split
# =============================================================================

DECHAZAL_DS1 = {
    "101", "106", "108", "109", "112", "114", "115", "116",
    "118", "119", "122", "124", "201", "203", "205", "207",
    "208", "209", "215", "220", "223", "230",
}

DECHAZAL_DS2 = {
    "100", "103", "105", "111", "113", "117", "121", "123",
    "200", "202", "210", "212", "213", "214", "219", "221",
    "222", "228", "231", "232", "233", "234",
}


# =============================================================================
# Patient mapping
# =============================================================================

def record_to_patient_id(record_id: str) -> str:
    """
    MIT-BIH contains 48 records from 47 subjects.

    Records 201 and 202 belong to the same subject.
    All other records are treated as separate subjects.
    """
    if record_id in {"201", "202"}:
        return "patient_201_202"

    return f"patient_{record_id}"


# =============================================================================
# Download
# =============================================================================

def download_mitbih(dataset_dir: Path) -> None:
    dataset_dir.mkdir(parents=True, exist_ok=True)

    existing_headers = list(dataset_dir.glob("*.hea"))

    if len(existing_headers) >= 48:
        print("MIT-BIH already appears to be downloaded.")
        return

    print("Downloading MIT-BIH from PhysioNet...")

    wfdb.dl_database(
        "mitdb",
        dl_dir=str(dataset_dir),
        records="all",
        annotators="all",
        keep_subdirs=False,
        overwrite=False,
    )

    print("Download complete.")


# =============================================================================
# Channel selection
# =============================================================================

def select_channel(record, record_id: str, channel_mode: str):
    signal_names = [str(x).upper() for x in record.sig_name]

    if channel_mode.lower() == "first":
        return 0, record.sig_name[0]

    if channel_mode.upper() == "MLII":
        if "MLII" not in signal_names:
            return None, None

        channel_index = signal_names.index("MLII")
        return channel_index, record.sig_name[channel_index]

    raise ValueError(
        "CHANNEL_MODE must be 'MLII' or 'first'."
    )


# =============================================================================
# Resampling
# =============================================================================

def resample_ecg(signal: np.ndarray, source_fs: float, target_fs: int):
    """
    Polyphase resampling.

    MIT-BIH:
        360 -> 250 Hz

    simplifies exactly to:
        up = 25
        down = 36
    """
    source_fs_int = int(round(source_fs))

    gcd = np.gcd(source_fs_int, target_fs)

    up = target_fs // gcd
    down = source_fs_int // gcd

    signal_resampled = resample_poly(
        signal,
        up=up,
        down=down,
    )

    return signal_resampled.astype(np.float32), up, down


def map_annotation_sample(
    original_sample: int,
    source_fs: float,
    target_fs: int,
) -> int:
    return int(
        round(
            original_sample
            * target_fs
            / source_fs
        )
    )


# =============================================================================
# Beat extraction
# =============================================================================

def extract_centered_window(
    signal: np.ndarray,
    center: int,
    n_samples: int,
):
    """
    Extract exactly n_samples centered around center.

    For n_samples=187:
        93 samples before R
        R at position 93
        93 samples after R
    """
    left = n_samples // 2
    right = n_samples - left

    start = center - left
    stop = center + right

    if start < 0 or stop > len(signal):
        return None

    beat = signal[start:stop]

    if len(beat) != n_samples:
        return None

    return beat.astype(np.float32)


# =============================================================================
# Build complete beat dataset
# =============================================================================

def build_complete_dataset(
    dataset_dir: Path,
    target_fs: int,
    n_samples: int,
    channel_mode: str,
):
    record_ids = sorted(
        path.stem
        for path in dataset_dir.glob("*.hea")
        if path.stem.isdigit()
    )

    if len(record_ids) == 0:
        raise RuntimeError(
            "No MIT-BIH records found. "
            "Check DATASET_DIR or enable DOWNLOAD_DATABASE."
        )

    print(f"\nFound {len(record_ids)} records.")

    beats = []
    labels = []
    metadata = []

    global_beat_id = 0
    skipped_records = []

    for record_number, record_id in enumerate(record_ids, start=1):
        print(
            f"[{record_number:02d}/{len(record_ids):02d}] "
            f"Record {record_id}",
            end="",
        )

        record_path = str(dataset_dir / record_id)

        record = wfdb.rdrecord(record_path)
        annotation = wfdb.rdann(record_path, "atr")

        source_fs = float(record.fs)

        channel_index, channel_name = select_channel(
            record,
            record_id,
            channel_mode,
        )

        if channel_index is None:
            print(
                f" -> skipped "
                f"(available channels: {record.sig_name})"
            )

            skipped_records.append(
                {
                    "record_id": record_id,
                    "reason": "MLII unavailable",
                    "channels": ",".join(record.sig_name),
                }
            )

            continue

        signal = record.p_signal[:, channel_index].astype(np.float32)

        signal_resampled, up, down = resample_ecg(
            signal,
            source_fs,
            target_fs,
        )

        record_count = 0

        for annotation_index, (sample, symbol) in enumerate(
            zip(annotation.sample, annotation.symbol)
        ):
            if symbol not in AAMI_MAP:
                continue

            aami_class = AAMI_MAP[symbol]
            label = CLASS_TO_INDEX[aami_class]

            resampled_rpeak = map_annotation_sample(
                int(sample),
                source_fs,
                target_fs,
            )

            beat = extract_centered_window(
                signal_resampled,
                center=resampled_rpeak,
                n_samples=n_samples,
            )

            if beat is None:
                continue

            beats.append(beat)
            labels.append(label)

            metadata.append(
                {
                    "beat_id": global_beat_id,
                    "record_id": record_id,
                    "patient_id": record_to_patient_id(record_id),
                    "annotation_index": annotation_index,
                    "original_symbol": symbol,
                    "aami_class": aami_class,
                    "label": label,
                    "source_fs": source_fs,
                    "target_fs": target_fs,
                    "original_rpeak_sample": int(sample),
                    "resampled_rpeak_sample": resampled_rpeak,
                    "channel_name": channel_name,
                    "channel_index": channel_index,
                }
            )

            global_beat_id += 1
            record_count += 1

        print(
            f" -> {record_count:,} beats | "
            f"{channel_name} | "
            f"{source_fs:g}->{target_fs} Hz"
        )

    X = np.stack(beats).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    metadata_df = pd.DataFrame(metadata)

    print("\nDataset construction complete.")
    print("X:", X.shape)
    print("y:", y.shape)

    if skipped_records:
        print("\nSkipped records:")
        for row in skipped_records:
            print(
                f"  {row['record_id']}: "
                f"{row['reason']} "
                f"[{row['channels']}]"
            )

    return X, y, metadata_df, pd.DataFrame(skipped_records)


# =============================================================================
# Random beat-wise split
# =============================================================================

def random_split(
    y: np.ndarray,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int,
):
    if not np.isclose(
        train_frac + val_frac + test_frac,
        1.0,
    ):
        raise ValueError(
            "TRAIN_FRAC + VAL_FRAC + TEST_FRAC must equal 1."
        )

    indices = np.arange(len(y))

    train_idx, temp_idx = train_test_split(
        indices,
        test_size=(1.0 - train_frac),
        random_state=seed,
        stratify=y,
    )

    temp_y = y[temp_idx]

    relative_test_frac = test_frac / (val_frac + test_frac)

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=relative_test_frac,
        random_state=seed,
        stratify=temp_y,
    )

    return train_idx, val_idx, test_idx


# =============================================================================
# Strict patient-independent split
# =============================================================================

def inter_patient_split(
    metadata_df: pd.DataFrame,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int,
):
    """
    Strict patient-independent split.

    Records 201 and 202 share a patient_id and therefore will always
    remain in the same partition.

    This is NOT the canonical de Chazal DS1/DS2 split.
    """
    if not np.isclose(
        train_frac + val_frac + test_frac,
        1.0,
    ):
        raise ValueError(
            "TRAIN_FRAC + VAL_FRAC + TEST_FRAC must equal 1."
        )

    indices = np.arange(len(metadata_df))
    groups = metadata_df["patient_id"].values

    splitter1 = GroupShuffleSplit(
        n_splits=1,
        train_size=train_frac,
        random_state=seed,
    )

    train_idx, temp_idx = next(
        splitter1.split(
            indices,
            groups=groups,
        )
    )

    temp_groups = groups[temp_idx]

    relative_val_frac = val_frac / (val_frac + test_frac)

    splitter2 = GroupShuffleSplit(
        n_splits=1,
        train_size=relative_val_frac,
        random_state=seed + 1,
    )

    val_relative_idx, test_relative_idx = next(
        splitter2.split(
            temp_idx,
            groups=temp_groups,
        )
    )

    val_idx = temp_idx[val_relative_idx]
    test_idx = temp_idx[test_relative_idx]

    return train_idx, val_idx, test_idx


# =============================================================================
# Canonical de Chazal split
# =============================================================================

def dechazal_split(
    metadata_df: pd.DataFrame,
    seed: int,
):
    """
    Canonical MIT-BIH DS1 -> train/validation
    and DS2 -> test.

    DS1 and DS2 each contain 22 records.

    Since the original protocol has no validation partition, 15% of
    DS1 records are held out as validation at RECORD level.

    Important:
        Records 201 and 202 belong to the same person but are placed in
        DS1 and DS2 in the historical protocol. Therefore this protocol
        is not perfectly patient-independent.
    """
    record_ids = metadata_df["record_id"].astype(str)

    ds1_mask = record_ids.isin(DECHAZAL_DS1).values
    ds2_mask = record_ids.isin(DECHAZAL_DS2).values

    ds1_idx = np.where(ds1_mask)[0]
    test_idx = np.where(ds2_mask)[0]

    ds1_records = metadata_df.iloc[ds1_idx]["record_id"].values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed,
    )

    train_relative_idx, val_relative_idx = next(
        splitter.split(
            ds1_idx,
            groups=ds1_records,
        )
    )

    train_idx = ds1_idx[train_relative_idx]
    val_idx = ds1_idx[val_relative_idx]

    return train_idx, val_idx, test_idx


# =============================================================================
# Normalization
# =============================================================================

def normalize_splits(
    X_train: np.ndarray,
    X_val: np.ndarray,
    X_test: np.ndarray,
    mode: str,
):
    mode = mode.lower()

    normalization_info = {
        "mode": mode,
    }

    if mode == "none":
        return X_train, X_val, X_test, normalization_info

    if mode == "per_beat_zscore":
        def normalize(X):
            mean = X.mean(axis=1, keepdims=True)
            std = X.std(axis=1, keepdims=True)
            return ((X - mean) / (std + 1e-8)).astype(np.float32)

        X_train = normalize(X_train)
        X_val = normalize(X_val)
        X_test = normalize(X_test)

        return X_train, X_val, X_test, normalization_info

    if mode == "global_zscore":
        mean = float(X_train.mean())
        std = float(X_train.std())

        X_train = ((X_train - mean) / (std + 1e-8)).astype(np.float32)
        X_val = ((X_val - mean) / (std + 1e-8)).astype(np.float32)
        X_test = ((X_test - mean) / (std + 1e-8)).astype(np.float32)

        normalization_info["train_mean"] = mean
        normalization_info["train_std"] = std

        return X_train, X_val, X_test, normalization_info

    raise ValueError(
        "NORMALIZATION must be one of: "
        "'none', 'per_beat_zscore', 'global_zscore'."
    )


# =============================================================================
# Diagnostics
# =============================================================================

def print_split_summary(
    name: str,
    y: np.ndarray,
    metadata: pd.DataFrame,
):
    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)

    print(f"Beats:    {len(y):,}")
    print(f"Records:  {metadata['record_id'].nunique()}")
    print(f"Patients: {metadata['patient_id'].nunique()}")

    print("\nClass distribution:")

    for class_index in sorted(CLASS_TO_INDEX.values()):
        class_name = INDEX_TO_CLASS[class_index]
        count = int(np.sum(y == class_index))
        percentage = count / len(y) if len(y) else 0

        print(
            f"  {class_name}: "
            f"{count:7,d} "
            f"({percentage:6.2%})"
        )

    print("\nRecords:")
    print(
        ", ".join(
            sorted(
                metadata["record_id"].astype(str).unique()
            )
        )
    )


def verify_patient_independence(
    train_meta: pd.DataFrame,
    val_meta: pd.DataFrame,
    test_meta: pd.DataFrame,
):
    train_patients = set(train_meta["patient_id"])
    val_patients = set(val_meta["patient_id"])
    test_patients = set(test_meta["patient_id"])

    train_val_overlap = train_patients & val_patients
    train_test_overlap = train_patients & test_patients
    val_test_overlap = val_patients & test_patients

    print("\nPatient overlap:")
    print("  train <-> val: ", train_val_overlap)
    print("  train <-> test:", train_test_overlap)
    print("  val   <-> test:", val_test_overlap)

    return (
        len(train_val_overlap) == 0
        and len(train_test_overlap) == 0
        and len(val_test_overlap) == 0
    )


# =============================================================================
# Save outputs
# =============================================================================

def save_dataset(
    output_dir: Path,
    X: np.ndarray,
    y: np.ndarray,
    metadata_df: pd.DataFrame,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    X_train = X[train_idx]
    X_val = X[val_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    train_meta = metadata_df.iloc[train_idx].copy()
    val_meta = metadata_df.iloc[val_idx].copy()
    test_meta = metadata_df.iloc[test_idx].copy()

    train_meta["split"] = "train"
    val_meta["split"] = "val"
    test_meta["split"] = "test"

    X_train, X_val, X_test, normalization_info = normalize_splits(
        X_train,
        X_val,
        X_test,
        NORMALIZATION,
    )

    # -------------------------------------------------------------------------
    # Six primary NumPy files
    # -------------------------------------------------------------------------

    np.save(output_dir / "X_train.npy", X_train)
    np.save(output_dir / "X_val.npy", X_val)
    np.save(output_dir / "X_test.npy", X_test)

    np.save(output_dir / "y_train.npy", y_train)
    np.save(output_dir / "y_val.npy", y_val)
    np.save(output_dir / "y_test.npy", y_test)

    # -------------------------------------------------------------------------
    # Metadata
    # -------------------------------------------------------------------------

    all_split_metadata = pd.concat(
        [train_meta, val_meta, test_meta],
        ignore_index=True,
    )

    train_meta.to_csv(
        output_dir / "metadata_train.csv",
        index=False,
    )

    val_meta.to_csv(
        output_dir / "metadata_val.csv",
        index=False,
    )

    test_meta.to_csv(
        output_dir / "metadata_test.csv",
        index=False,
    )

    all_split_metadata.to_csv(
        output_dir / "metadata_all_splits.csv",
        index=False,
    )

    # Save exact indices into the original complete dataset.
    np.savez(
        output_dir / "split_indices.npz",
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

    if SAVE_FULL_DATASET:
        np.save(
            output_dir / "X_all.npy",
            X,
        )

        np.save(
            output_dir / "y_all.npy",
            y,
        )

        metadata_df.to_csv(
            output_dir / "metadata_all.csv",
            index=False,
        )

    with open(
        output_dir / "class_mapping.json",
        "w",
    ) as f:
        json.dump(
            CLASS_TO_INDEX,
            f,
            indent=2,
        )

    with open(
        output_dir / "normalization.json",
        "w",
    ) as f:
        json.dump(
            normalization_info,
            f,
            indent=2,
        )

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        train_meta,
        val_meta,
        test_meta,
    )


# =============================================================================
# Main
# =============================================================================

def main():
    print("=" * 70)
    print("MIT-BIH DATASET BUILDER")
    print("=" * 70)

    print(f"Dataset path:  {DATASET_DIR}")
    print(f"Output path:   {OUTPUT_DIR}")
    print(f"Sampling rate: {TARGET_FS} Hz")
    print(f"Beat length:   {N_SAMPLES}")
    print(f"Channel mode:  {CHANNEL_MODE}")
    print(f"Normalization: {NORMALIZATION}")
    print(f"Split mode:    {SPLIT_MODE}")

    # -------------------------------------------------------------------------
    # Download
    # -------------------------------------------------------------------------

    if DOWNLOAD_DATABASE:
        download_mitbih(DATASET_DIR)

    # -------------------------------------------------------------------------
    # Build all annotated beats
    # -------------------------------------------------------------------------

    X, y, metadata_df, skipped_df = build_complete_dataset(
        dataset_dir=DATASET_DIR,
        target_fs=TARGET_FS,
        n_samples=N_SAMPLES,
        channel_mode=CHANNEL_MODE,
    )

    # -------------------------------------------------------------------------
    # Split
    # -------------------------------------------------------------------------

    if SPLIT_MODE == "random":
        train_idx, val_idx, test_idx = random_split(
            y=y,
            train_frac=TRAIN_FRAC,
            val_frac=VAL_FRAC,
            test_frac=TEST_FRAC,
            seed=RANDOM_SEED,
        )

    elif SPLIT_MODE == "inter_patient":
        train_idx, val_idx, test_idx = inter_patient_split(
            metadata_df=metadata_df,
            train_frac=TRAIN_FRAC,
            val_frac=VAL_FRAC,
            test_frac=TEST_FRAC,
            seed=RANDOM_SEED,
        )

    elif SPLIT_MODE == "dechazal":
        train_idx, val_idx, test_idx = dechazal_split(
            metadata_df=metadata_df,
            seed=RANDOM_SEED,
        )

    else:
        raise ValueError(
            "SPLIT_MODE must be "
            "'random', 'inter_patient', or 'dechazal'."
        )

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------

    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        train_meta,
        val_meta,
        test_meta,
    ) = save_dataset(
        output_dir=OUTPUT_DIR,
        X=X,
        y=y,
        metadata_df=metadata_df,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
    )

    if len(skipped_df) > 0:
        skipped_df.to_csv(
            OUTPUT_DIR / "skipped_records.csv",
            index=False,
        )

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    print_split_summary(
        "Training",
        y_train,
        train_meta,
    )

    print_split_summary(
        "Validation",
        y_val,
        val_meta,
    )

    print_split_summary(
        "Test",
        y_test,
        test_meta,
    )

    if SPLIT_MODE == "inter_patient":
        is_independent = verify_patient_independence(
            train_meta,
            val_meta,
            test_meta,
        )

        if not is_independent:
            raise RuntimeError(
                "Patient leakage detected!"
            )

        print(
            "\nStrict patient-independence check: PASSED"
        )

    if SPLIT_MODE == "dechazal":
        print(
            "\nNOTE: canonical de Chazal DS1/DS2 is being used."
        )
        print(
            "Records 201 and 202 belong to the same subject but are "
            "historically assigned to opposite partitions."
        )

    # -------------------------------------------------------------------------
    # Configuration log
    # -------------------------------------------------------------------------

    config = {
        "database": "MIT-BIH Arrhythmia Database",
        "target_fs": TARGET_FS,
        "n_samples": N_SAMPLES,
        "channel_mode": CHANNEL_MODE,
        "normalization": NORMALIZATION,
        "split_mode": SPLIT_MODE,
        "train_fraction": TRAIN_FRAC,
        "val_fraction": VAL_FRAC,
        "test_fraction": TEST_FRAC,
        "random_seed": RANDOM_SEED,
        "class_mapping": CLASS_TO_INDEX,
    }

    with open(
        OUTPUT_DIR / "dataset_config.json",
        "w",
    ) as f:
        json.dump(
            config,
            f,
            indent=2,
        )

    print("\n" + "=" * 70)
    print("DONE")
    print("=" * 70)

    print("\nPrimary files:")
    for filename in [
        "X_train.npy",
        "X_val.npy",
        "X_test.npy",
        "y_train.npy",
        "y_val.npy",
        "y_test.npy",
    ]:
        print(f"  {OUTPUT_DIR / filename}")

    print("\nShapes:")
    print("X_train:", X_train.shape)
    print("X_val:  ", X_val.shape)
    print("X_test: ", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_val:  ", y_val.shape)
    print("y_test: ", y_test.shape)


if __name__ == "__main__":
    main()